# 170 — Model Behavior Profiling

Profile overall model behavior: prediction confidence, attention patterns,
computation budget allocation, position difficulty, and summary statistics.

## Why This Matters

Model behavior provides tools for systematic analysis and comparison of transformer internals. These diagnostics help identify unexpected behaviors, compare architectures, and build a comprehensive picture of how the model processes information.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.model_behavior_profiling import (
    prediction_confidence_profile,
    attention_pattern_profile,
    computation_budget_profile,
    position_difficulty_profile,
    model_summary_stats,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.arange(15)

In [ ]:
result = prediction_confidence_profile(model, tokens)
print(f"Mean confidence: {result['mean_confidence']:.4f}  Mean entropy: {result['mean_entropy']:.3f}\n")
for p in result['per_position']:
    bar = '#' * int(p['top_probability'] * 30)
    print(f"  pos {p['position']:2d}: tok={p['top_token']:3d}  "
          f"p={p['top_probability']:.4f}  H={p['entropy']:.3f}  {bar}")

In [ ]:
result = attention_pattern_profile(model, tokens)
for l in result['per_layer']:
    heads = '  '.join(f"H{h['head']}:H={h['mean_entropy']:.2f}" for h in l['per_head'])
    print(f"Layer {l['layer']}: mean_H={l['mean_entropy']:.3f}  "
          f"sparse={l['n_sparse_heads']}/{len(l['per_head'])}  [{heads}]")

In [ ]:
result = computation_budget_profile(model, tokens)
print(f"Total computation: {result['total_computation']:.4f}\n")
for b in result['per_layer']:
    attn_bar = '#' * int(b['attn_fraction'] * 20)
    mlp_bar = '#' * int((1 - b['attn_fraction']) * 20)
    print(f"Layer {b['layer']}: attn={b['attn_budget']:.3f} [{attn_bar}]  "
          f"mlp={b['mlp_budget']:.3f} [{mlp_bar}]  total_frac={b['fraction_of_total']:.3f}")

In [ ]:
result = position_difficulty_profile(model, tokens)
for p in result['per_position']:
    bar = '#' * int(p['difficulty'] * 30)
    print(f"  #{p['difficulty_rank']}: pos {p['position']:2d}  "
          f"diff={p['difficulty']:.4f}  conf={p['confidence']:.4f}  gap={p['top2_gap']:.4f}  {bar}")

In [ ]:
stats = model_summary_stats(model, tokens)
for k, v in stats.items():
    print(f"  {k}: {v}")